### About Project
#### TripCraft Batam - Notebook 03: Penyusunan Itinerary

Mengubah hasil pencarian jadi **jadwal harian** lengkap dengan jam dan tempat makan. Termasuk:
- Sinonim (kata informal -> kata baku)
- Koreksi typo (salah ketik diperbaiki)
- Scheduler (slot makan & wisata otomatis)

In [1]:
import pandas as pd
import re

In [2]:
data = pd.read_csv("data/cleaned/batam_cleaned.csv")

# Skor kualitas + label Recommended (deterministik, SAMA dengan build_recommend.py & NB02) -- bukan output ML, untuk hindari kebocoran data
import numpy as np
def _norm(s):
    s = s.astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)
data["Quality_Score"] = 0.6*_norm(data["Rating"]) + 0.4*_norm(np.log1p(data["Jumlah_Ulasan"]))
data["Recommended"]   = (data["Quality_Score"] >= data["Quality_Score"].median()).astype(int)

In [3]:
def preprocess(t):
    return re.sub(r"\s+"," ", re.sub(r"[^a-z\s]","", str(t).lower())).strip()

data["search_clean"] = data["Search_Text"].apply(preprocess)

### TF-IDF + skor relevansi

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TF-IDF yang sama dengan yang diekspor ke website (build_recommend.py)
tfidf = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True)
matriks = tfidf.fit_transform(data["search_clean"])
vocab = set(tfidf.vocabulary_.keys())

### Sinonim & Koreksi Typo
Supaya input user yang informal atau salah ketik tetap dimengerti.

In [5]:
# kamus sinonim
sinonim = {"makan":"kuliner","jajan":"kuliner","nyemil":"kuliner",
           "nginep":"hotel","penginapan":"hotel","santai":"pantai",
           "ngopi":"kafe","kopi":"kafe","pedes":"pedas","shopping":"belanja"}

# koreksi typo: cari kata di vocab yang beda 1 huruf
def edits1(word):
    huruf = "abcdefghijklmnopqrstuvwxyz"
    pecah = [(word[:i], word[i:]) for i in range(len(word)+1)]
    hapus = [a+b[1:] for a,b in pecah if b]
    ganti = [a+c+b[1:] for a,b in pecah if b for c in huruf]
    sisip = [a+c+b for a,b in pecah for c in huruf]
    return set(hapus + ganti + sisip)

def koreksi(word):
    if word in vocab: return word
    kandidat = edits1(word) & vocab
    return min(kandidat, key=len) if kandidat else word

def proses_query(q):
    hasil = []
    for w in preprocess(q).split():
        w = sinonim.get(w, w)
        w = koreksi(w)
        hasil.append(w)
    return " ".join(hasil)

In [6]:
# coba
proses_query("mau jajan seafod di nagoa")

'mau kuliner seafood di nagoya'

### Skor relevansi tiap tempat terhadap query

In [7]:
def relevansi(query):
    qv = tfidf.transform([proses_query(query)])
    return cosine_similarity(qv, matriks).ravel()

### Scheduler (penjadwalan berbasis KEDEKATAN lokasi)



In [8]:
import numpy as np

# Jarak dua titik dalam km (Haversine) -> untuk mengelompokkan tempat yang BENAR-BENAR berdekatan.
def jarak_km(la1, lo1, la2, lo2):
    R = 6371.0
    dla = np.radians(la2 - la1); dlo = np.radians(lo2 - lo1)
    a = np.sin(dla/2)**2 + np.cos(np.radians(la1))*np.cos(np.radians(la2))*np.sin(dlo/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Slot waktu (rentang jam) + tipe: 'a' = aktivitas/atraksi, 'm' = makan
SLOTS = [("09.00-11.00","a"), ("11.15-12.45","a"), ("13.00-14.00","m"),
         ("14.30-16.30","a"), ("16.45-18.15","a"), ("18.30-20.00","m")]

# Kata kunci -> kategori, untuk bonus skor sesuai minat user (sama seperti di website)
CATMAP = {"kuliner":"Kuliner","seafood":"Kuliner","kafe":"Kuliner","mie":"Kuliner",
          "foodcourt":"Kuliner","pujasera":"Kuliner","belanja":"Belanja","mall":"Belanja",
          "religi":"Religi","masjid":"Religi","vihara":"Religi","wisata":"Wisata",
          "pantai":"Wisata","budaya":"Wisata","museum":"Wisata","sejarah":"Wisata",
          "golf":"Alam","hotel":"Hotel"}

def skor_semua(query):
    """Skor tiap tempat = relevansi (CBF/TF-IDF) + bonus rating + bonus kategori sesuai minat."""
    pq = proses_query(query) if query.strip() else ""
    rel = relevansi(query) if pq else np.zeros(len(data))
    minat = {CATMAP[w] for w in pq.split() if w in CATMAP}
    d = data.copy()
    d["rel"]  = rel
    d["skor"] = rel + d["Rating"]/100 + d["Category"].isin(minat) * 0.3
    return d

def _urut_dekat(pool, anchor):
    """Urutkan dari yang TERDEKAT ke anchor; jika jarak mirip (<1.2km), dahulukan Recommended lalu skor."""
    pool = pool.copy()
    pool["jarak"] = jarak_km(anchor["Lat"], anchor["Long"], pool["Lat"], pool["Long"])
    pool["bin"]   = (pool["jarak"] / 1.2).round()
    return pool.sort_values(["bin","Recommended","skor"], ascending=[True, False, False])

def buat_itinerary(query, jumlah_hari=1):
    d = skor_semua(query)
    # tema yang diminta (selain makan/hotel) -> aktivitas WAJIB ikut tema ini (mis. "ibadah & budaya" -> hanya Religi/Wisata, TIDAK ada mall). Sama persis dengan website.
    pq = proses_query(query) if query.strip() else ""
    minat_akt = {CATMAP[w] for w in pq.split() if w in CATMAP and CATMAP[w] not in ("Kuliner","Hotel")}
    if minat_akt:
        atraksi = lambda df: df[df["Category"].isin(minat_akt)]             # patuh tema
    else:
        atraksi = lambda df: df[~df["Category"].isin(["Kuliner","Hotel"])]   # tanpa tema -> semua atraksi
    kuliner = lambda df: df[df["Category"] == "Kuliner"]

    dipakai = set(); jadwal = {}
    for hari in range(1, jumlah_hari + 1):
        tersisa = d[~d["Place_Id"].isin(dipakai)]
        if tersisa.empty: break

        # ANCHOR: dari 8 atraksi skor tertinggi, pilih yang lingkungannya PADAT (>=3 tempat dalam 6km)
        top = atraksi(tersisa).sort_values(["Recommended","skor"], ascending=False).head(8)
        anchor = None
        for _, p in top.iterrows():
            lain = tersisa[tersisa["Place_Id"] != p["Place_Id"]]
            if (jarak_km(p["Lat"], p["Long"], lain["Lat"], lain["Long"]) <= 6).sum() >= 3:
                anchor = p; break
        if anchor is None:
            anchor = top.iloc[0] if len(top) else tersisa.sort_values("skor", ascending=False).iloc[0]

        plan = []; jml_kat = {}; n_makan = 0
        for jam, tipe in SLOTS:
            sisa = d[~d["Place_Id"].isin(dipakai)]
            pilih = None
            if tipe == "m":
                # makan: kuliner TERDEKAT dari anchor, maksimal ~15km; kalau tak ada yang dekat -> slot dilewati
                km = _urut_dekat(kuliner(sisa), anchor)
                km = km[km["jarak"] <= 15]
                if len(km): pilih = km.iloc[0]
            else:
                # aktivitas: atraksi TERDEKAT (<=6km) dulu, utamakan kategori baru (variasi)
                ak  = _urut_dekat(atraksi(sisa), anchor)
                dkt = ak[ak["jarak"] <= 6]
                cand = dkt[dkt["Category"].apply(lambda c: jml_kat.get(c,0) < 1)]
                if not len(cand): cand = dkt[dkt["Category"].apply(lambda c: jml_kat.get(c,0) < 2)]
                if not len(cand): cand = dkt
                if not len(cand): cand = ak[ak["Category"].apply(lambda c: jml_kat.get(c,0) < 1)]
                if not len(cand): cand = ak
                if len(cand): pilih = cand.iloc[0]

            if pilih is not None:
                label = ""
                if tipe == "m":
                    n_makan += 1
                    label = "Makan Siang" if n_makan == 1 else "Makan Malam"
                plan.append((jam, label, pilih))
                dipakai.add(pilih["Place_Id"])
                jml_kat[pilih["Category"]] = jml_kat.get(pilih["Category"], 0) + 1
        # hari tanpa SATU PUN aktivitas (cuma slot makan) = stok atraksi se-tema habis
        # -> berhenti, jangan buat "hari hantu" isi makan doang (SAMA dengan website)
        ada_aktivitas = any(not label for _, label, _ in plan)
        if plan and ada_aktivitas:
            jadwal[hari] = plan
        else:
            break
    return jadwal

### Demo: itinerary lengkap

In [9]:
def tampilkan(jadwal):
    for hari, plan in jadwal.items():
        print(f"\nHARI {hari}")
        for jam, label, poi in plan:
            tag = f"[{label.upper()}] " if label else ""
            rec = "  (Recommended)" if poi["Recommended"] else ""
            print(f"  {jam}  {tag}{poi['Place_Name']} ({poi['Category']}) - rating {poi['Rating']}{rec}")

jadwal = buat_itinerary("wisata pantai dan kuliner seafood", 2)
tampilkan(jadwal)


HARI 1
  09.00-11.00  Batam Adventure Park (Wisata) - rating 4.7  (Recommended)
  11.15-12.45  Panbil Nature Reserve Eco Edu Park (Wisata) - rating 4.3
  13.00-14.00  [MAKAN SIANG] Morning Bakery (Kuliner) - rating 4.3
  14.30-16.30  Museum Batam Raja Ali Haji (Wisata) - rating 4.5
  16.45-18.15  Mega Wisata Ocarina Batam (Wisata) - rating 4.2
  18.30-20.00  [MAKAN MALAM] Anchor Café & Roastery (Kuliner) - rating 4.6  (Recommended)

HARI 2
  09.00-11.00  Taman Rusa Sekupang (Wisata) - rating 4.5  (Recommended)
  11.15-12.45  Pantai Tanjung Pinggir (Wisata) - rating 4.0
  13.00-14.00  [MAKAN SIANG] Morning Bakery Tiban (Kuliner) - rating 4.1
  14.30-16.30  Wisata Bukit Harimau (Wisata) - rating 4.5
  16.45-18.15  Pesona Alam Tangga Seribu (Wisata) - rating 4.0
  18.30-20.00  [MAKAN MALAM] CW Coffe, Coffe & Eatery (Kuliner) - rating 4.1


### Demo: user tidak tahu mau ke mana (query kosong)
Sistem otomatis kasih rencana populer yang bervariasi.

In [10]:
jadwal = buat_itinerary("", 1)
tampilkan(jadwal)


HARI 1
  09.00-11.00  Masjid Sultan Mahmud Riayat Syah (Religi) - rating 4.8  (Recommended)
  11.15-12.45  Batam Hills Golf Resort (Alam) - rating 4.4
  13.00-14.00  [MAKAN SIANG] Kopi Lain Hati Batu Aji (Kuliner) - rating 4.4
  14.30-16.30  Taman Rusa Sekupang (Wisata) - rating 4.5  (Recommended)
  16.45-18.15  Panbil Mall (Belanja) - rating 4.5  (Recommended)
  18.30-20.00  [MAKAN MALAM] CW Coffe, Coffe & Eatery (Kuliner) - rating 4.1


## Ringkasan
Pipeline lengkap:
- Query user -> sinonim & koreksi typo -> skor relevansi (cosine similarity)
- Scheduler menyusun jadwal: slot makan diisi kuliner, slot wisata diisi tempat bervariasi
- Output: itinerary harian dengan jam

Logika ini yang dipakai di web app TripCraft Batam.